# FineWeb-2 ind_Latn

## Dataset

**FineWeb-2** adalah korpus web multibahasa dari Hugging Face yang isinya teks halaman web
yang sudah dibersihkan. Versi yang dipakai di tugas ini adalah konfigurasi **`ind_Latn`**,
yaitu dokumen berbahasa Indonesia dengan aksara Latin.

| | |
|---|---|
| Nama | FineWeb-2 |
| Konfigurasi | `ind_Latn` (Bahasa Indonesia, Latin) |
| Sumber | https://huggingface.co/datasets/HuggingFaceFW/fineweb-2 |
| Config `ind_Latn` | https://huggingface.co/datasets/HuggingFaceFW/fineweb-2?config=ind_Latn |
| Berkas yang dipakai | `data/ind_Latn/train/000_00000.parquet` |
| Baris | 1.200.000 |
| Ukuran | 1.681,4 MB |

Satu baris berisi satu dokumen: `text` (isi halaman), `url`, `date` (waktu crawl), `id`,
`dump` (batch crawl Common Crawl), `language`, `language_score`, `minhash_cluster_size`, dan
kolom pendukung lain. Jumlahnya 11 kolom.

Shard `000_00000.parquet` aslinya 3.149.000 baris / 4,8 GB. Yang dipakai di sini 1,2 juta
dokumen pertama, disimpan sebagai `data/raw/fineweb2_indo_1_2jt.parquet` lewat
`src/download_dataset.py`. Cara unduh ulang ada di `data/README.md`.

**Syarat tugas**: minimal 500 MB atau lebih dari 1.000.000 baris. Dataset ini memenuhi
keduanya (1.681,4 MB dan 1.200.000 baris).

## Library

In [1]:
from pathlib import Path

import duckdb
import polars as pl

print("polars :", pl.__version__)
print("duckdb :", duckdb.__version__)

polars : 1.44.2
duckdb : 1.5.5


## Baca data

Path dihitung dari folder kerja, supaya notebook bisa dijalankan dari root repo
maupun dari dalam folder `notebooks/`.

In [2]:
akar = Path.cwd()
while not (akar / "data").is_dir() and akar != akar.parent:
    akar = akar.parent

DATA_PATH = akar / "data" / "raw" / "fineweb2_indo_1_2jt.parquet"

print(DATA_PATH)
print(f"ukuran : {DATA_PATH.stat().st_size / 1e6:,.1f} MB")

/home/jovyan/work/data/raw/fineweb2_indo_1_2jt.parquet
ukuran : 1,681.4 MB


## Skema kolom

Dibaca dengan `scan_parquet` (lazy), jadi isi berkas belum dimuat.

In [3]:
data = pl.scan_parquet(DATA_PATH)

for nama, tipe in data.collect_schema().items():
    print(f"{nama:22} {tipe}")

text                   String
id                     String
dump                   String
url                    String
date                   String
file_path              String
language               String
language_score         Float64
language_script        String
minhash_cluster_size   Int64
top_langs              String


In [4]:
jumlah_baris = data.select(pl.len()).collect().item()
jumlah_kolom = len(data.collect_schema())

print(f"baris : {jumlah_baris:,}")
print(f"kolom : {jumlah_kolom}")
print()
print("syarat tugas (>1.000.000 baris) :", "terpenuhi" if jumlah_baris > 1_000_000 else "BELUM")

baris : 1,200,000
kolom : 11

syarat tugas (>1.000.000 baris) : terpenuhi


## Arti kolom

| Kolom | Isi |
|---|---|
| `text` | isi teks halaman web |
| `id` | penanda dokumen, berbentuk `urn:uuid:...` |
| `dump` | batch crawl Common Crawl, misalnya `CC-MAIN-2024-10` |
| `url` | alamat halaman sumber |
| `date` | waktu crawl, format `YYYY-MM-DDTHH:MM:SSZ` |
| `file_path` | lokasi berkas asal di penyimpanan Common Crawl |
| `language` | label bahasa, untuk dataset ini `ind` |
| `language_score` | skor keyakinan deteksi bahasa, 0 sampai 1 |
| `language_script` | jenis aksara, untuk dataset ini `Latn` |
| `minhash_cluster_size` | jumlah dokumen dalam klaster MinHash yang sama |
| `top_langs` | daftar bahasa lain yang terdeteksi pada dokumen yang sama |

In [5]:
data.select(["id", "date", "url", "language", "language_score", "minhash_cluster_size"]).head(5).collect()

id,date,url,language,language_score,minhash_cluster_size
str,str,str,str,f64,i64
"""<urn:uuid:a218c3db-7870-4149-8…","""2013-05-19T22:07:28Z""","""http://amiratthemovies.wordpre…","""ind""",0.985925,17
"""<urn:uuid:139c36f7-0e4c-4205-b…","""2013-05-19T21:58:29Z""","""http://azmara-chan.blogspot.co…","""ind""",0.989975,16
"""<urn:uuid:d4303450-21f0-41fc-9…","""2013-05-19T21:58:08Z""","""http://efekphotoshop.com/tutor…","""ind""",0.947492,38
"""<urn:uuid:e51e0aea-c9a7-4a09-9…","""2013-05-19T22:07:36Z""","""http://gomilyuner.com/ringkasa…","""ind""",0.985864,1
"""<urn:uuid:4dcd2475-4cc0-4196-8…","""2013-05-19T21:50:30Z""","""http://jellygamat-gold.com/unc…","""ind""",0.97137,1


In [6]:
contoh = data.select("text").head(2).collect()

for i, teks in enumerate(contoh["text"], 1):
    print(f"--- dokumen {i}, {len(teks)} karakter ---")
    print(teks[:400].replace("\n", " "))
    print()

--- dokumen 1, 6488 karakter ---
Diangkat dari cerita pendek berjudul Ghost Walker karya Ian MacKenzie Jeffers, yang bersama sutradara Joe Carnahan (The A-Team, 2010) juga menyusun naskah cerita untuk film ini, The Grey mengisahkan mengenai sekelompok pekerja pertambangan minyak di hari terakhir mereka bekerja. Setelah selama lima minggu terdampar di dinginnya hamparan luas Alaska, para pekerja tersebut akhirnya dapat kembali ke 

--- dokumen 2, 1172 karakter ---
Seiring dengan dirilisnya Firefox 21 kemarin, ada berbagai fitur baru yang ditanamkan. Salah satunya adalah Firefox Health Report (FHR). Fitur ini adalah fitur yang akan mentracking berbagai data seperti startup time, total running time, dan jumlah crash yang terjadi. Data tersebut akan secara otomatis dikirimkan ke Mozilla untuk digunakan sebagai acuan dalam pengembangan Firefox berikutnya. Fitur



## Cek nilai kosong

In [7]:
kolom = data.collect_schema().names()
null_per_kolom = data.select([pl.col(k).null_count().alias(k) for k in kolom]).collect().row(0)

for k, n in zip(kolom, null_per_kolom):
    print(f"{k:22} {n:>9,}  ({n / jumlah_baris * 100:5.2f}%)")

text                           0  ( 0.00%)
id                             0  ( 0.00%)
dump                           0  ( 0.00%)
url                            0  ( 0.00%)
date                           0  ( 0.00%)
file_path                      0  ( 0.00%)
language                       0  ( 0.00%)
language_score                 0  ( 0.00%)
language_script                0  ( 0.00%)
minhash_cluster_size           0  ( 0.00%)
top_langs                      0  ( 0.00%)


## Nilai unik

In [8]:
data.select([
    pl.col("language").n_unique().alias("language"),
    pl.col("language_script").n_unique().alias("language_script"),
    pl.col("dump").n_unique().alias("dump"),
    pl.col("url").n_unique().alias("url"),
]).collect()

language,language_script,dump,url
u32,u32,u32,u32
1,1,96,1199358


In [9]:
sebaran_dump = (
    data.group_by("dump")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .collect()
)
sebaran_dump

dump,jumlah
str,u32
"""CC-MAIN-2023-40""",27679
"""CC-MAIN-2023-50""",25587
"""CC-MAIN-2024-10""",23830
"""CC-MAIN-2024-18""",22544
"""CC-MAIN-2022-49""",22040
…,…
"""CC-MAIN-2015-06""",2602
"""CC-MAIN-2016-18""",2527
"""CC-MAIN-2015-40""",1995


## Rentang tanggal

Kolom `date` masih teks, diubah ke datetime dulu.

In [10]:
data_tanggal = data.with_columns(
    pl.col("date").str.to_datetime("%Y-%m-%dT%H:%M:%SZ", strict=False)
)

rentang = data_tanggal.select([
    pl.col("date").min().alias("crawl_awal"),
    pl.col("date").max().alias("crawl_akhir"),
    pl.col("date").dt.year().n_unique().alias("jumlah_tahun"),
]).collect()
rentang

crawl_awal,crawl_akhir,jumlah_tahun
datetime[μs],datetime[μs],u32
2013-05-18 11:11:32,2024-04-25 15:44:13,12


In [11]:
per_tahun = (
    data_tanggal.with_columns(pl.col("date").dt.year().alias("tahun"))
    .group_by("tahun")
    .agg(pl.len().alias("jumlah_dokumen"))
    .sort("tahun")
    .collect()
)
per_tahun

tahun,jumlah_dokumen
i32,u32
2013,5224
2014,40744
2015,27422
2016,40889
2017,166504
…,…
2020,140321
2021,175676
2022,111017


## SUMMARIZE (DuckDB)

In [12]:
con = duckdb.connect()
con.execute("SET threads = 4")

profil = con.execute(f"""
    SUMMARIZE SELECT
        id, dump, url, date, language,
        language_score, language_script, minhash_cluster_size
    FROM read_parquet('{DATA_PATH.as_posix()}')
""").pl()

profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""VARCHAR""","""<urn:uuid:00000357-6767-43ce-8…","""<urn:uuid:fffff95e-119f-41e2-a…",972354,null,null,null,null,null,1200000,0.00
"""dump""","""VARCHAR""","""CC-MAIN-2013-20""","""CC-MAIN-2024-18""",92,null,null,null,null,null,1200000,0.00
"""url""","""VARCHAR""","""http://00-r.com/2021/09/13/nut…","""https://zzzyy.blogspot.com/201…",1010285,null,null,null,null,null,1200000,0.00
"""date""","""VARCHAR""","""2013-05-18T11:11:32Z""","""2024-04-25T15:44:13Z""",1129017,null,null,null,null,null,1200000,0.00
"""language""","""VARCHAR""","""ind""","""ind""",1,null,null,null,null,null,1200000,0.00
"""language_score""","""DOUBLE""","""0.6845871806144714""","""0.9996840953826904""",458376,"""0.9582589115315676""","""0.04543719713284284""","""0.9510137018182487""","""0.9735234325301048""","""0.9849285071174975""",1200000,0.00
"""language_script""","""VARCHAR""","""Latn""","""Latn""",1,null,null,null,null,null,1200000,0.00
"""minhash_cluster_size""","""BIGINT""","""1""","""1286170""",4341,"""48.07024333333333""","""2059.600098702298""","""3""","""8""","""18""",1200000,0.00


## Sebaran language_score

In [13]:
data.select([
    pl.col("language_score").min().alias("min"),
    pl.col("language_score").quantile(0.05).alias("p05"),
    pl.col("language_score").median().alias("median"),
    pl.col("language_score").quantile(0.95).alias("p95"),
    pl.col("language_score").max().alias("max"),
    (pl.col("language_score") < 0.5).sum().alias("di_bawah_0_5"),
]).collect()

min,p05,median,p95,max,di_bawah_0_5
f64,f64,f64,f64,f64,u32
0.684587,0.864643,0.973501,0.993516,0.999684,0


## Panjang teks

In [14]:
panjang = data.select(pl.col("text").str.len_chars().alias("panjang"))

panjang.select([
    pl.col("panjang").min().alias("min"),
    pl.col("panjang").quantile(0.25).alias("q1"),
    pl.col("panjang").median().alias("median"),
    pl.col("panjang").quantile(0.75).alias("q3"),
    pl.col("panjang").max().alias("max"),
    pl.col("panjang").mean().round(0).alias("rata_rata"),
]).collect()

min,q1,median,q3,max,rata_rata
u32,f64,f64,f64,u32,f64
273,1638.0,2674.0,4375.0,737704,3885.0


In [15]:
print("dokumen di bawah 200 karakter :", panjang.filter(pl.col("panjang") < 200).select(pl.len()).collect().item())
print("dokumen di bawah 500 karakter :", panjang.filter(pl.col("panjang") < 500).select(pl.len()).collect().item())

dokumen di bawah 200 karakter : 0
dokumen di bawah 500 karakter : 27945


## Petunjuk duplikat

`minhash_cluster_size` = jumlah dokumen dalam klaster MinHash yang sama.

In [16]:
klaster = (
    data.group_by("minhash_cluster_size")
    .agg(pl.len().alias("jumlah_dokumen"))
    .sort("minhash_cluster_size")
    .collect()
)
klaster.head(10)

minhash_cluster_size,jumlah_dokumen
i64,u32
1,123316
2,114694
3,87740
4,84501
5,74703
6,60384
7,51670
8,54734
9,38431


In [17]:
total_berklaster = klaster.filter(pl.col("minhash_cluster_size") > 1)["jumlah_dokumen"].sum()
print(f"dokumen yang masuk klaster berisi lebih dari satu : {total_berklaster:,}")
print(f"bagian dari seluruh data                          : {total_berklaster / jumlah_baris * 100:.1f}%")

dokumen yang masuk klaster berisi lebih dari satu : 1,076,684
bagian dari seluruh data                          : 89.7%


## Catatan

- 1.200.000 baris, 11 kolom, tidak ada null.
- `language` isinya cuma `ind`, `language_script` cuma `Latn`. Ada 96 batch crawl, 1.199.358 URL unik.
- Crawl 2013-05-18 sampai 2024-04-25. Naik tajam dari 2016 ke 2017 (40.889 jadi 166.504), puncak 2021 (175.676). 2024 cuma 46.374 karena crawlnya berhenti April.
- `language_score` terendah 0,6846, jadi ambang 0,5 tidak membuang apa pun. Ambang untuk cleaning harus diambil dari angka lain.
- Panjang teks 273 sampai 737.704 karakter, median 2.674. Yang di bawah 500 karakter ada 27.945 dokumen (2,3%).
- 89,7% dokumen ada di klaster MinHash yang isinya lebih dari satu dokumen. URL hampir semua berbeda, jadi kemungkinan isi disalin antar situs. Perlu dicek di notebook 02.